#### 1. Loading CISA and MITRE data
The classified CISA events and MITRE ATT&CK techniques are loaded to identify potential relationships between them.

In [1]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data/processed")

events = pd.read_csv(data_dir / "cisa_classified_events.csv")
techniques = pd.read_csv(data_dir / "mitre_techniques.csv")

print("CISA events:", len(events))
print("MITRE techniques:", len(techniques))

display(events[["inject_id", "event_description", "threat_category"]].head())
display(techniques[["technique_id", "name", "description"]].head())

CISA events: 66
MITRE techniques: 706


,inject_id,event_description,threat_category
0,HC-INJ-87,The Cybersecurity and Infrastructure Security ...,No Confirmed Incident
1,HC-INJ-89,<Organization> employees receive e-mail notifi...,Phishing
2,HC-INJ-117,Manufacturer’s updates are pushed for installa...,No Confirmed Incident
3,HC-INJ-119,A third-party electronic medical record (EMR) ...,Vulnerability
4,HC-INJ-126,Radiology reports an issue with their Magnetic...,System Disruption


,technique_id,name,description
0,T1055.011,Extra Window Memory Injection,Adversaries may inject malicious code into a p...
1,T1053.005,Scheduled Task,Adversaries may abuse the Windows Task Schedul...
2,T1205.002,Socket Filters,Adversaries may attach filters to a network so...
3,T1560.001,Archive via Utility,Adversaries may use utilities to compress and/...
4,T1021.005,VNC,Adversaries may use [Valid Accounts](https://a...


#### 2. Comparing CISA events with MITRE techniques
TF-IDF and cosine similarity are used to identify MITRE techniques whose descriptions are most similar to the CISA events.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Prepare text for comparison
event_texts = events["event_description"].fillna("")

technique_texts = (
    techniques["name"].fillna("")
    + " "
    + techniques["description"].fillna("")
)

# Convert all descriptions into TF-IDF vectors
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf = vectorizer.fit_transform(
    pd.concat([event_texts, technique_texts], ignore_index=True)
)

# Separate CISA and MITRE vectors
event_vectors = tfidf[:len(events)]
technique_vectors = tfidf[len(events):]

# Calculate similarity between each event and each technique
similarities = cosine_similarity(event_vectors, technique_vectors)

print("Similarity matrix:", similarities.shape)

Similarity matrix: (66, 706)
